# Reasoning with Sampling — MATH 实验（Colab）


In [ ]:
#@title 安装依赖
%pip install -q transformers accelerate pandas tqdm bitsandbytes


In [ ]:
#@title 只下载 MATH500 数据（JSON 较大，仍从网络拉取）
import os
import urllib.request

REPO = os.environ.get("RWS_GITHUB_REPO", "aakaran/reasoning-with-sampling")
REF = os.environ.get("RWS_GITHUB_REF", "main")
url = f"https://raw.githubusercontent.com/{REPO}/{REF}/llm_experiments/data/MATH500.json"
MATH500_PATH = "/content/MATH500.json"

print("Downloading", url)
urllib.request.urlretrieve(url, MATH500_PATH)
print("Saved:", MATH500_PATH, "size:", os.path.getsize(MATH500_PATH), "bytes")


In [ ]:
#@title 工具函数与采样逻辑（与 llm_experiments 中代码一致，内联在笔记本内）
# --- constants (llm_experiments/constants.py) ---
PROMPT = "Can you solve the following math problem? "
BASE = " Put your final answer within \\boxed{{}}."
COT = " Please reason step by step, and put your final answer within \\boxed{{}}."
COT_ALT = " Please explain your reasoning with a detailed, step-by-step solution, and present your final answer within \\boxed{{}}."
GPQA_QUERY_TEMPLATE = "Answer the following multiple choice question. The last line of your response should be of the following format: '\\boxed{{$LETTER}}' (without quotes) where LETTER is one of ABCD (ex. '\\boxed{{A}}'). Think step by step before answering.\n\n{Question}\n\nA) {A}\nB) {B}\nC) {C}\nD) {D}"

# --- grader_utils/parse_utils.py ---
def remove_boxed(s):
    left = "\\boxed{"
    try:
        assert s[:len(left)] == left
        assert s[-1] == "}"
        return s[len(left):-1]
    except Exception:
        return None

def last_boxed_only_string(string):
    idx = string.rfind("\\boxed")
    if idx < 0:
        idx = string.rfind("\\fbox")
        if idx < 0:
            return None
    i = idx
    right_brace_idx = None
    num_left_braces_open = 0
    while i < len(string):
        if string[i] == "{":
            num_left_braces_open += 1
        if string[i] == "}":
            num_left_braces_open -= 1
            if num_left_braces_open == 0:
                right_brace_idx = i
                break
        i += 1
    if right_brace_idx is None:
        return None
    return string[idx : right_brace_idx + 1]

def parse_answer(input_str):
    boxed = last_boxed_only_string(input_str or "")
    return remove_boxed(boxed) if boxed is not None else None

def normalize_math_answer(answer):
    if answer is None:
        return None
    s = str(answer).strip()
    boxed = last_boxed_only_string(s)
    if boxed is not None:
        s = remove_boxed(boxed) or s
    replacements = {
        "\\left": "", "\\right": "", "\\!": "", "\\,": "",
        "\\;": "", "\\ ": "", "\\text": "", "\\mathrm": "",
    }
    for old, new in replacements.items():
        s = s.replace(old, new)
    return "".join(s.split()).strip(".$")

def answers_match(prediction, target):
    pred_norm = normalize_math_answer(prediction)
    target_norm = normalize_math_answer(target)
    return pred_norm is not None and target_norm is not None and pred_norm == target_norm

# --- power_samp_utils.py (MATH 所需) ---
import random
import numpy as np
import torch
import torch.nn as nn
import transformers
from torch.nn import functional as F
from tqdm.auto import tqdm

def load_text_processor(model_str):
    processor = None
    tokenizer = transformers.AutoTokenizer.from_pretrained(model_str, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    return processor, tokenizer

def encode_text_prompt(processor, tokenizer, text, device):
    inputs = tokenizer(text, return_tensors="pt")
    return inputs.to(device)

def load_generation_model(model_str, load_in_4bit=False, load_in_8bit=False):
    load_kwargs = {"torch_dtype": "auto", "device_map": "auto", "trust_remote_code": True}
    if load_in_4bit:
        load_kwargs["load_in_4bit"] = True
    elif load_in_8bit:
        load_kwargs["load_in_8bit"] = True
    return transformers.AutoModelForCausalLM.from_pretrained(model_str, **load_kwargs).eval()

def infer_model_device(model):
    return next(model.parameters()).device

def model_vocab_size(model):
    return model.get_input_embeddings().num_embeddings

def print_cuda_memory(label):
    if not torch.cuda.is_available():
        return
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"{label}: cuda allocated={allocated:.2f} GiB reserved={reserved:.2f} GiB")

def forward_text_only(model, input_ids, attention_mask):
    return model(input_ids=input_ids, attention_mask=attention_mask)

def generate_text_only(model, input_ids, attention_mask, **generate_kwargs):
    return model.generate(input_ids=input_ids, attention_mask=attention_mask, **generate_kwargs)

# Power sampling: sample from p^{alpha}; 1/alpha 由 temperature 传入
class AutoregressiveSampler:
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        text_config = getattr(self.model.config, "text_config", self.model.config)
        self.block_size = getattr(text_config, "max_position_embeddings", 32768)

    @torch.no_grad()
    def next_token(self, prefix):
        device = self.device
        torch_prefix = torch.tensor([prefix], dtype=torch.long, device=device)
        attention_mask = torch.ones_like(torch_prefix)
        prefix_cond = (
            torch_prefix
            if torch_prefix.size(1) <= self.block_size
            else torch_prefix[:, -self.block_size :]
        )
        mask_cond = attention_mask[:, -prefix_cond.size(1):]
        output = forward_text_only(self.model, prefix_cond, mask_cond)
        logits = output.logits[0, -1, :]
        probs = F.softmax(logits, dim=-1)
        return torch.log(probs)

def normalize(dist):
    return F.softmax(dist, dim=-1)

def dist_product(logit_p, logit_q):
    return logit_p + logit_q

def dist_temp_scale(logit_p, temp):
    return logit_p * torch.tensor(1 / temp, dtype=logit_p.dtype, device=logit_p.device)

def naive_temp(p, context, temp, seq_len):
    c = len(context)
    device = p.device
    tokenizer = p.tokenizer
    input_ids = torch.tensor([context], dtype=torch.long, device=device)
    output = generate_text_only(
        p.model,
        input_ids=input_ids,
        attention_mask=torch.ones_like(input_ids),
        max_new_tokens=seq_len - c,
        do_sample=True,
        temperature=temp,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        return_dict_in_generate=True,
        output_scores=True,
        output_logits=True,
    )
    unscaled_logits = torch.stack(output.logits, dim=0)
    scaled_logits = torch.stack(output.scores, dim=0)
    tokens = output.sequences[0][c:]
    prop = output.sequences[0].tolist()
    assert len(tokens) == unscaled_logits.shape[0] == scaled_logits.shape[0]
    idx = tokens.view(unscaled_logits.shape[0], 1, 1)
    log_probs_unnorm = (1 / temp * torch.gather(F.log_softmax(unscaled_logits, dim=-1), -1, idx)).view(
        -1
    ).tolist()
    log_probs_norm = torch.gather(F.log_softmax(scaled_logits, dim=-1), -1, idx).view(-1).tolist()
    assert len(tokens) == len(log_probs_unnorm) == len(log_probs_norm)
    return prop, log_probs_norm, log_probs_unnorm

def max_swap(p, context, temp, mcmc_steps, max_new_tokens, block_num=16):
    c = len(context)
    print("Temp:", temp)
    gen = []
    if context is not None:
        gen = context.copy()
    log_probs_norm = []
    log_probs_unnorm = []
    assert max_new_tokens % block_num == 0
    jump_size = int(max_new_tokens // block_num)
    print("max_new_tokens:", max_new_tokens, "jump_size:", jump_size)
    attempts = 0
    acceptances = 0
    for _ in tqdm(range(block_num)):
        gen, lp_norm, lp_unnorm = naive_temp(p, gen, temp=temp, seq_len=jump_size + len(gen))
        log_probs_norm.extend(lp_norm)
        log_probs_unnorm.extend(lp_unnorm)
        for _ in tqdm(range(mcmc_steps)):
            attempts += 1
            t = len(gen)
            idx = random.randint(c, t - 1)
            prop, log_prob_prop, target_log_prob_prop = naive_temp(
                p, gen[:idx], temp=temp, seq_len=t
            )
            s = len(prop)
            assert len(log_prob_prop) == s - idx
            assert len(target_log_prob_prop) == s - idx
            log_prob_cur = log_probs_norm.copy()[idx - c : s - c]
            target_log_prob_cur = log_probs_unnorm.copy()[idx - c : s - c]
            log_r = sum(target_log_prob_prop) - sum(target_log_prob_cur)
            if log_r > 0:
                acceptances += 1
                gen = prop.copy()
                log_probs_norm[idx - c :] = log_prob_prop.copy()
                log_probs_unnorm[idx - c :] = target_log_prob_prop.copy()
        if p.tokenizer.eos_token_id in gen[c:]:
            eos_idx = c + gen[c:].index(p.tokenizer.eos_token_id)
            gen = gen[: eos_idx + 1]
            log_probs_norm = log_probs_norm[: eos_idx - c + 1]
            log_probs_unnorm = log_probs_unnorm[: eos_idx - c + 1]
            return gen, log_probs_norm, log_probs_unnorm, acceptances / max(1, attempts)
    return gen, log_probs_norm, log_probs_unnorm, acceptances / max(1, attempts)

def mcmc_power_samp(p, context, temp, mcmc_steps, max_new_tokens, block_num=16, local_moves=False):
    c = len(context)
    print("alpha:", 1 / temp)
    gen = []
    if context is not None:
        gen = context.copy()
    log_probs_norm = []
    log_probs_unnorm = []
    assert max_new_tokens % block_num == 0
    jump_size = int(max_new_tokens // block_num)
    print("max_new_tokens:", max_new_tokens, "jump_size:", jump_size)
    attempts = 0
    acceptances = 0
    for _ in tqdm(range(block_num)):
        gen, lp_norm, lp_unnorm = naive_temp(p, gen, temp=temp, seq_len=jump_size + len(gen))
        log_probs_norm.extend(lp_norm)
        log_probs_unnorm.extend(lp_unnorm)
        for _ in tqdm(range(mcmc_steps)):
            attempts += 1
            t = len(gen)
            if local_moves:
                # Windowed MH: only resample the most recent jump_size tokens.
                # Cuts work per step from O(t) to O(jump_size); changes the chain
                # to block-local rather than global, matching the speculative version.
                lo = max(c, t - jump_size)
            else:
                lo = c
            idx = random.randint(lo, t - 1)
            prop, log_prob_prop, target_log_prob_prop = naive_temp(
                p, gen[:idx], temp=temp, seq_len=t
            )
            s = len(prop)
            assert len(log_prob_prop) == s - idx
            assert len(target_log_prob_prop) == s - idx
            log_prob_cur = log_probs_norm.copy()[idx - c : s - c]
            target_log_prob_cur = log_probs_unnorm.copy()[idx - c : s - c]
            log_r = (
                sum(target_log_prob_prop)
                + sum(log_prob_cur)
                - sum(target_log_prob_cur)
                - sum(log_prob_prop)
            )
            if np.random.rand() < np.exp(log_r):
                acceptances += 1
                gen = prop.copy()
                log_probs_norm[idx - c :] = log_prob_prop.copy()
                log_probs_unnorm[idx - c :] = target_log_prob_prop.copy()
        if p.tokenizer.eos_token_id in gen[c:]:
            eos_idx = c + gen[c:].index(p.tokenizer.eos_token_id)
            gen = gen[: eos_idx + 1]
            log_probs_norm = log_probs_norm[: eos_idx - c + 1]
            log_probs_unnorm = log_probs_unnorm[: eos_idx - c + 1]
            return gen, log_probs_norm, log_probs_unnorm, acceptances / max(1, attempts)
    return gen, log_probs_norm, log_probs_unnorm, acceptances / max(1, attempts)

def format_prompt(question, model, tokenizer, cot=True):
    if model in ("qwen", "qwen_small", "qwen_math", "qwen_math_small"):
        format_str = PROMPT + question
        format_str += COT if cot else BASE
    elif model in ("qwen_instruct_small", "qwen3_small", "qwen3_8b", "qwen_math_grpo", "phi_grpo", "phi", "tulu"):
        content_str = PROMPT + question
        content_str += COT if cot else BASE
        answer_context = [{"role": "user", "content": content_str}]
        format_str = tokenizer.apply_chat_template(
            answer_context, tokenize=False, add_generation_prompt=True,enable_thinking=False
        )
    else:
        raise ValueError(f"Unsupported model key for format_prompt: {model}")
    return format_str


In [ ]:
#@title 配置 + 运行 MATH（power sampling）
import os
import json
import random
import time
import gc
import torch
import pandas as pd
import transformers
from tqdm.auto import tqdm

# 可改配置
MODEL_KEY = "qwen3_8b"
BATCH_IDX = 0
SEED = 0
MCMC_STEPS = 4
TEMPERATURE = 0.25
MAX_NEW_TOKENS = 1024
USE_COT = True  # 是否用 CoT；勿用名 COT，会覆盖工具格里的 COT 字符串常
SAVE_DIR = "/content/rws_out"
MAX_PROBLEMS = 10  # 小规模正式实验；完整 shard 用 None
MCMC_LOCAL_MOVES = True  # fix #2: only resample the most recent jump_size tokens per MH step (~5–10× faster, block-local chain)

MODEL_REPOS = {
    "qwen": "Qwen/Qwen2.5-7B",
    "qwen_math": "Qwen/Qwen2.5-Math-7B",
    "qwen3_small": "Qwen/Qwen3-0.6B",
    "qwen3_8b": "Qwen/Qwen3-8B",
    "qwen_math_grpo": "stellalisy/rethink_rlvr_reproduce-ground_truth-qwen2.5_math_7b-lr5e-7-kl0.00-step150",
    "phi": "microsoft/Phi-3.5-mini-instruct",
    "tulu": "allenai/Llama-3.1-Tulu-3-8B-DPO",
}
if MODEL_KEY not in MODEL_REPOS:
    raise ValueError(f"Unknown MODEL_KEY: {MODEL_KEY}")
model_str = MODEL_REPOS[MODEL_KEY]

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("未检测到 GPU，请启用 GPU 运行时。")

random.seed(SEED)
torch.manual_seed(SEED)

json_path = "/content/MATH500.json"
if not os.path.isfile(json_path):
    raise FileNotFoundError("先运行上一格下载 MATH500.json")

with open(json_path, "r") as f:
    dataset = json.load(f)

start = 100 * BATCH_IDX
end = 100 * (BATCH_IDX + 1)
if MAX_PROBLEMS is not None:
    end = min(start + int(MAX_PROBLEMS), end)
print(f"shard [{start}, {end}) of MATH500, model={model_str} device={device}")

os.makedirs(SAVE_DIR, exist_ok=True)
out_dir = os.path.join(SAVE_DIR, MODEL_KEY)
os.makedirs(out_dir, exist_ok=True)

print("Loading processor/tokenizer and model (首次会下载权重)...")
processor, tokenizer = load_text_processor(model_str)
hf_model = load_generation_model(model_str)
device = infer_model_device(hf_model)
autoreg_sampler = AutoregressiveSampler(hf_model, tokenizer, device)
print("Model loaded.")

# Diagnostics.
_p = next(hf_model.parameters())
_attn = getattr(hf_model.config, "_attn_implementation", "?")
print(f"[diag] dtype={_p.dtype}  device={_p.device}  attn_impl={_attn}")
if torch.cuda.is_available():
    print(f"[diag] gpu={torch.cuda.get_device_name(0)}  cuda={torch.version.cuda}")

def sync_cuda():
    if str(device).startswith("cuda"):
        torch.cuda.synchronize()

results = []
for data in tqdm(dataset[start:end], desc="MATH power sampling"):
    question = data["prompt"]
    answer = data["answer"]
    input_text = format_prompt(question, MODEL_KEY, tokenizer, USE_COT)
    model_inputs = encode_text_prompt(processor, tokenizer, input_text, device)
    input_ids = model_inputs["input_ids"]
    prompt_len = input_ids.shape[1]
    prefx = [idx.item() for idx in input_ids[0]]

    sync_cuda()
    naive_start = time.perf_counter()
    naive_temp_output = hf_model.generate(
        **model_inputs, max_new_tokens=MAX_NEW_TOKENS, return_dict_in_generate=True,
        output_scores=True, do_sample=True, temperature=TEMPERATURE,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
    )
    sync_cuda()
    naive_seconds = time.perf_counter() - naive_start

    sync_cuda()
    std_start = time.perf_counter()
    std_output = hf_model.generate(
        **model_inputs, max_new_tokens=MAX_NEW_TOKENS, return_dict_in_generate=True,
        output_scores=True, do_sample=True,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
    )
    sync_cuda()
    std_seconds = time.perf_counter() - std_start

    sync_cuda()
    mcmc_start = time.perf_counter()
    mcmc_power_samp_output, _, _, acceptance_ratio = mcmc_power_samp(
        autoreg_sampler, prefx, TEMPERATURE, MCMC_STEPS,
        max_new_tokens=MAX_NEW_TOKENS, local_moves=MCMC_LOCAL_MOVES,
    )
    sync_cuda()
    mcmc_seconds = time.perf_counter() - mcmc_start

    naive_ids = naive_temp_output.sequences[0, prompt_len:].to("cpu")
    std_ids = std_output.sequences[0, prompt_len:].to("cpu")
    mcmc_generated_ids = mcmc_power_samp_output[prompt_len:]

    naive_c = tokenizer.decode(naive_ids, skip_special_tokens=True)
    std_c = tokenizer.decode(std_ids, skip_special_tokens=True)
    mcmc_c = tokenizer.decode(mcmc_generated_ids, skip_special_tokens=True)

    naive_answer = parse_answer(naive_c)
    std_answer = parse_answer(std_c)
    mcmc_answer = parse_answer(mcmc_c)

    results.append({
        "question": question,
        "correct_answer": answer,
        "correct_answer_normalized": normalize_math_answer(answer),
        "naive_completion": naive_c,
        "naive_answer": naive_answer,
        "naive_answer_normalized": normalize_math_answer(naive_answer),
        "naive_correct": answers_match(naive_answer, answer),
        "naive_tokens": len(naive_ids),
        "naive_seconds": naive_seconds,
        "naive_tokens_per_second": len(naive_ids) / max(naive_seconds, 1e-9),
        "std_completion": std_c,
        "std_answer": std_answer,
        "std_answer_normalized": normalize_math_answer(std_answer),
        "std_correct": answers_match(std_answer, answer),
        "std_tokens": len(std_ids),
        "std_seconds": std_seconds,
        "std_tokens_per_second": len(std_ids) / max(std_seconds, 1e-9),
        "mcmc_completion": mcmc_c,
        "mcmc_answer": mcmc_answer,
        "mcmc_answer_normalized": normalize_math_answer(mcmc_answer),
        "mcmc_correct": answers_match(mcmc_answer, answer),
        "mcmc_tokens": len(mcmc_generated_ids),
        "mcmc_seconds": mcmc_seconds,
        "mcmc_tokens_per_second": len(mcmc_generated_ids) / max(mcmc_seconds, 1e-9),
        "acceptance_ratio": acceptance_ratio,
    })
    print(
        "acceptance_ratio:", acceptance_ratio,
        "correct naive/std/mcmc:",
        answers_match(naive_answer, answer), answers_match(std_answer, answer), answers_match(mcmc_answer, answer),
    )

csv_name = f"{MODEL_KEY}_math_base_power_samp_{MCMC_STEPS}_{TEMPERATURE}_{BATCH_IDX}_{SEED}_{'local' if MCMC_LOCAL_MOVES else 'global'}.csv"
out_path = os.path.join(out_dir, csv_name)
df = pd.DataFrame(results)
df.to_csv(out_path, index=False)
metric_cols = ["naive_correct", "std_correct", "mcmc_correct", "acceptance_ratio", "naive_tokens_per_second", "std_tokens_per_second", "mcmc_tokens_per_second"]
display(df[metric_cols].mean(numeric_only=True).to_frame("mean").T)
print("Saved:", out_path)


## Speculative Power Sampling Prototype

This section keeps the baseline above untouched. It implements a fixed-block Metropolis-Hastings prototype where a draft model proposes candidate blocks and the target model scores those blocks in parallel under the sharpened target distribution.


In [ ]:
#@title Speculative power sampling utilities
import math

USE_TARGET_KV_CACHE = True
VALIDATE_TARGET_KV_CACHE_ONCE = True
TARGET_KV_CACHE_ATOL = 2e-3
TARGET_KV_CACHE_MODE = {
    "name": "full_mask+cache_position",
    "attention_mask": "full",
    "cache_position": True,
    "position_ids": False,
}

@torch.no_grad()
def score_continuation(model, context_ids, continuation_ids):
    """Return log p(continuation | context) for each batch row in one full forward pass."""
    if continuation_ids.numel() == 0:
        return torch.zeros(context_ids.shape[0], device=context_ids.device)

    full_ids = torch.cat([context_ids, continuation_ids], dim=1)
    attention_mask = torch.ones_like(full_ids)
    outputs = forward_text_only(model, full_ids, attention_mask)

    start = context_ids.shape[1] - 1
    end = full_ids.shape[1] - 1
    pred_logits = outputs.logits[:, start:end, :]
    log_probs = F.log_softmax(pred_logits.float(), dim=-1)
    token_log_probs = log_probs.gather(-1, continuation_ids.unsqueeze(-1)).squeeze(-1)
    return token_log_probs.sum(dim=1)


@torch.no_grad()
def build_target_prefix_cache(model, context_ids):
    """Cache target KV for the fixed prefix used by all proposals in one block."""
    attention_mask = torch.ones_like(context_ids)
    outputs = model(
        input_ids=context_ids,
        attention_mask=attention_mask,
        use_cache=True,
    )
    return {
        "past_key_values": outputs.past_key_values,
        "prefix_next_logits": outputs.logits[:, -1, :],
        "context_len": context_ids.shape[1],
        "attention_mask": attention_mask,
    }


@torch.no_grad()
def extend_target_prefix_cache(model, prefix_cache, new_token_ids, mode):
    """Append `new_token_ids` to an existing prefix cache without re-prefilling.

    Used at block-commit time so that the next block starts from a cache covering
    prompt + all committed blocks, avoiding the per-block O(context) prefill that
    `build_target_prefix_cache` would otherwise pay every iteration.
    """
    attention_mask = _cached_attention_mask(prefix_cache, new_token_ids, mode)
    positions = torch.arange(
        prefix_cache["context_len"],
        prefix_cache["context_len"] + new_token_ids.shape[1],
        device=new_token_ids.device,
    )
    kwargs = {
        "input_ids": new_token_ids,
        "attention_mask": attention_mask,
        "past_key_values": prefix_cache["past_key_values"],
        "use_cache": True,
    }
    if mode.get("cache_position"):
        kwargs["cache_position"] = positions
    if mode.get("position_ids"):
        kwargs["position_ids"] = positions.unsqueeze(0)
    out = model(**kwargs)

    new_context_len = prefix_cache["context_len"] + new_token_ids.shape[1]
    new_attention_mask = torch.cat(
        [prefix_cache["attention_mask"], torch.ones_like(new_token_ids)], dim=1
    )
    return {
        "past_key_values": out.past_key_values,
        "prefix_next_logits": out.logits[:, -1, :],
        "context_len": new_context_len,
        "attention_mask": new_attention_mask,
    }


def _target_cache_mode_candidates():
    return [
        {"name": "full_mask+cache_position", "attention_mask": "full", "cache_position": True, "position_ids": False},
        {"name": "full_mask+position_ids", "attention_mask": "full", "cache_position": False, "position_ids": True},
        {"name": "full_mask", "attention_mask": "full", "cache_position": False, "position_ids": False},
        {"name": "new_mask+cache_position", "attention_mask": "new", "cache_position": True, "position_ids": False},
        {"name": "new_mask+position_ids", "attention_mask": "new", "cache_position": False, "position_ids": True},
        {"name": "new_mask", "attention_mask": "new", "cache_position": False, "position_ids": False},
    ]


def _cached_attention_mask(prefix_cache, cont_input, mode):
    if mode.get("attention_mask") == "new":
        return torch.ones_like(cont_input)
    return torch.cat([prefix_cache["attention_mask"], torch.ones_like(cont_input)], dim=1)


@torch.no_grad()
def _cached_forward(model, prefix_cache, cont_input, mode):
    attention_mask = _cached_attention_mask(prefix_cache, cont_input, mode)
    positions = torch.arange(
        prefix_cache["context_len"],
        prefix_cache["context_len"] + cont_input.shape[1],
        device=cont_input.device,
    )
    kwargs = {
        "input_ids": cont_input,
        "attention_mask": attention_mask,
        "past_key_values": prefix_cache["past_key_values"],
        "use_cache": False,
    }
    if mode.get("cache_position"):
        kwargs["cache_position"] = positions
    if mode.get("position_ids"):
        kwargs["position_ids"] = positions.unsqueeze(0)
    try:
        return model(**kwargs)
    except (TypeError, ValueError, RuntimeError) as exc:
        # Some Transformers cache APIs reject one mask convention but accept another.
        raise exc


@torch.no_grad()
def score_continuation_cached(model, prefix_cache, continuation_ids, mode=None):
    """Return log p(continuation | cached context) without recomputing the prefix."""
    if mode is None:
        mode = TARGET_KV_CACHE_MODE
    if continuation_ids.numel() == 0:
        return torch.zeros(1, device=prefix_cache["prefix_next_logits"].device)

    first_token = continuation_ids[:, :1]
    first_log_probs = F.log_softmax(prefix_cache["prefix_next_logits"].float(), dim=-1)
    first_token_logprob = first_log_probs.gather(-1, first_token).squeeze(-1)

    if continuation_ids.shape[1] == 1:
        return first_token_logprob

    cont_input = continuation_ids[:, :-1]
    outputs = _cached_forward(model, prefix_cache, cont_input, mode)

    later_targets = continuation_ids[:, 1:]
    later_log_probs = F.log_softmax(outputs.logits.float(), dim=-1)
    later_token_logprobs = later_log_probs.gather(-1, later_targets.unsqueeze(-1)).squeeze(-1)
    return first_token_logprob + later_token_logprobs.sum(dim=1)


def validate_cached_score_once(model, context_ids, continuation_ids, atol=TARGET_KV_CACHE_ATOL):
    """Probe cache conventions and return the best matching mode, or None if unsafe."""
    uncached = score_continuation(model, context_ids, continuation_ids)
    prefix_cache = build_target_prefix_cache(model, context_ids)
    results = []
    for mode in _target_cache_mode_candidates():
        try:
            cached = score_continuation_cached(model, prefix_cache, continuation_ids, mode=mode)
            max_diff = (uncached - cached).abs().max().item()
            results.append((max_diff, mode))
            print(f"target cache validation {mode['name']} max_abs_diff={max_diff:.6f}")
        except Exception as exc:
            print(f"target cache validation {mode['name']} failed: {type(exc).__name__}: {exc}")
    if not results:
        print("target cache validation failed for all modes; disabling target KV cache")
        return None
    best_diff, best_mode = min(results, key=lambda item: item[0])
    if best_diff > atol:
        print(
            "target cache validation mismatch remains above tolerance; "
            f"best={best_mode['name']} max_abs_diff={best_diff:.6f}. Disabling target KV cache."
        )
        return None
    print(f"target cache validation selected {best_mode['name']} max_abs_diff={best_diff:.6f}")
    return best_mode

def _proposal_logprob_from_generate_scores(scores, continuation_ids):
    step_logps = []
    for step_scores, step_token in zip(scores, continuation_ids[0]):
        log_probs = F.log_softmax(step_scores.float(), dim=-1)
        step_logps.append(log_probs[0, step_token])
    if not step_logps:
        return torch.tensor(0.0, device=continuation_ids.device)
    return torch.stack(step_logps).sum()


@torch.no_grad()
def draft_propose(draft_model, tokenizer, context_ids, block_size, temperature=1.0):
    """Sample one continuation block from the draft model and return its proposal logprob."""
    out = generate_text_only(
        draft_model,
        input_ids=context_ids,
        attention_mask=torch.ones_like(context_ids),
        max_new_tokens=block_size,
        min_new_tokens=1,
        do_sample=True,
        temperature=temperature,
        return_dict_in_generate=True,
        output_scores=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    continuation_ids = out.sequences[:, context_ids.shape[1]:]
    draft_logprob = _proposal_logprob_from_generate_scores(out.scores, continuation_ids)
    return continuation_ids, draft_logprob


def speculative_power_step(
    target_model,
    draft_model,
    tokenizer,
    context_ids,
    old_block_ids,
    old_target_logprob,
    old_draft_logprob,
    alpha,
    block_size,
    draft_temperature=1.0,
    target_prefix_cache=None,
    target_cache_mode=None,
):
    """One independent-proposal MH update targeting p_target(block | context)^alpha."""
    new_block_ids, new_draft_logprob = draft_propose(
        draft_model, tokenizer, context_ids, block_size, temperature=draft_temperature
    )
    if target_prefix_cache is None:
        new_target_logprob = score_continuation(target_model, context_ids, new_block_ids).squeeze(0)
    else:
        new_target_logprob = score_continuation_cached(
            target_model, target_prefix_cache, new_block_ids, mode=target_cache_mode
        ).squeeze(0)

    log_accept_ratio = (
        alpha * new_target_logprob
        + old_draft_logprob
        - alpha * old_target_logprob
        - new_draft_logprob
    )
    log_accept_prob = torch.minimum(
        log_accept_ratio,
        torch.tensor(0.0, device=context_ids.device, dtype=log_accept_ratio.dtype),
    )
    accepted = torch.log(torch.rand((), device=context_ids.device)) < log_accept_prob

    if bool(accepted.item()):
        return new_block_ids, new_target_logprob, new_draft_logprob, True, float(log_accept_ratio.item())
    return old_block_ids, old_target_logprob, old_draft_logprob, False, float(log_accept_ratio.item())


def run_speculative_power_sampling(
    target_model,
    draft_model,
    tokenizer,
    context_ids,
    alpha,
    block_size,
    mcmc_steps_per_block,
    max_new_tokens,
    draft_temperature=1.0,
    use_target_kv_cache=USE_TARGET_KV_CACHE,
    validate_target_kv_cache_once=VALIDATE_TARGET_KV_CACHE_ONCE,
):
    """Generate by appending the final MH state for each fixed-size block."""
    generated = []
    context = context_ids
    block_stats = []
    total_attempts = 0
    total_acceptances = 0
    cache_validated = False
    selected_cache_mode = TARGET_KV_CACHE_MODE

    # Prefill once for the prompt; we extend rather than rebuild after each commit.
    target_prefix_cache = (
        build_target_prefix_cache(target_model, context) if use_target_kv_cache else None
    )

    max_blocks = math.ceil(max_new_tokens / block_size)
    for block_idx in tqdm(range(max_blocks), desc="Speculative power blocks"):
        remaining = max_new_tokens - len(generated)
        current_block_size = min(block_size, remaining)
        if current_block_size <= 0:
            break

        old_block, old_draft_logprob = draft_propose(
            draft_model, tokenizer, context, current_block_size, temperature=draft_temperature
        )
        if target_prefix_cache is None:
            old_target_logprob = score_continuation(target_model, context, old_block).squeeze(0)
        else:
            if validate_target_kv_cache_once and not cache_validated:
                selected_cache_mode = validate_cached_score_once(target_model, context, old_block)
                cache_validated = True
                if selected_cache_mode is None:
                    use_target_kv_cache = False
                    target_prefix_cache = None
            if target_prefix_cache is None:
                old_target_logprob = score_continuation(target_model, context, old_block).squeeze(0)
            else:
                old_target_logprob = score_continuation_cached(
                    target_model, target_prefix_cache, old_block, mode=selected_cache_mode
                ).squeeze(0)

        block_acceptances = 0
        last_log_accept_ratio = None
        for _ in range(mcmc_steps_per_block):
            total_attempts += 1
            old_block, old_target_logprob, old_draft_logprob, accepted, last_log_accept_ratio = speculative_power_step(
                target_model=target_model,
                draft_model=draft_model,
                tokenizer=tokenizer,
                context_ids=context,
                old_block_ids=old_block,
                old_target_logprob=old_target_logprob,
                old_draft_logprob=old_draft_logprob,
                alpha=alpha,
                block_size=current_block_size,
                draft_temperature=draft_temperature,
                target_prefix_cache=target_prefix_cache,
                target_cache_mode=selected_cache_mode,
            )
            if accepted:
                total_acceptances += 1
                block_acceptances += 1

        block_tokens = old_block[0].tolist()
        generated.extend(block_tokens)
        context = torch.cat([context, old_block], dim=1)

        # Advance the prefix cache by this committed block. One forward pass over
        # `block_size` tokens, vs the per-block prefill over the full context that
        # the rebuild approach would pay.
        if use_target_kv_cache and target_prefix_cache is not None:
            target_prefix_cache = extend_target_prefix_cache(
                target_model, target_prefix_cache, old_block, selected_cache_mode
            )

        block_stats.append({
            "block_idx": block_idx,
            "block_tokens": len(block_tokens),
            "block_acceptance_rate": block_acceptances / max(1, mcmc_steps_per_block),
            "target_logprob": float(old_target_logprob.item()),
            "draft_logprob": float(old_draft_logprob.item()),
            "last_log_accept_ratio": last_log_accept_ratio,
            "used_target_kv_cache": target_prefix_cache is not None,
        })

        if tokenizer.eos_token_id in block_tokens:
            eos_pos = generated.index(tokenizer.eos_token_id)
            generated = generated[:eos_pos + 1]
            break

    return generated, {
        "acceptance_ratio": total_acceptances / max(1, total_attempts),
        "attempts": total_attempts,
        "acceptances": total_acceptances,
        "blocks": block_stats,
        "used_target_kv_cache": use_target_kv_cache,
        "target_kv_cache_mode": selected_cache_mode["name"] if selected_cache_mode is not None else None,
    }






In [ ]:
#@title 配置 + 运行 MATH（speculative power sampling prototype）
import os
import json
import random
import time
import torch
import pandas as pd
import transformers
from tqdm.auto import tqdm

# 可改配置
SPEC_TARGET_MODEL_KEY = "qwen3_8b"
SPEC_DRAFT_MODEL_KEY = "qwen3_small"
SPEC_BATCH_IDX = 0
SPEC_SEED = 0
SPEC_ALPHA = 4.0
SPEC_BLOCK_SIZE = 64
SPEC_MCMC_STEPS_PER_BLOCK = 4
SPEC_MAX_NEW_TOKENS = 1024
SPEC_DRAFT_TEMPERATURE = 1.0
SPEC_TARGET_LOAD_IN_4BIT = False
SPEC_TARGET_LOAD_IN_8BIT = False
SPEC_DRAFT_LOAD_IN_4BIT = False
SPEC_DRAFT_LOAD_IN_8BIT = False
SPEC_USE_TARGET_KV_CACHE = True
SPEC_VALIDATE_TARGET_KV_CACHE_ONCE = True
FREE_BASELINE_MODEL_BEFORE_SPEC = True
SPEC_USE_COT = True
SPEC_SAVE_DIR = "/content/rws_out"
SPEC_MAX_PROBLEMS = MAX_PROBLEMS


SPEC_MODEL_REPOS = {
    "qwen": "Qwen/Qwen2.5-7B",
    "qwen_small": "Qwen/Qwen2.5-0.5B",
    "qwen_instruct_small": "Qwen/Qwen2.5-0.5B-Instruct",
    "qwen_math": "Qwen/Qwen2.5-Math-7B",
    "qwen_math_small": "Qwen/Qwen2.5-Math-1.5B",
    "qwen3_small": "Qwen/Qwen3-0.6B",
    "qwen3_8b": "Qwen/Qwen3-8B",
}

target_model_str = SPEC_MODEL_REPOS[SPEC_TARGET_MODEL_KEY]
draft_model_str = SPEC_MODEL_REPOS[SPEC_DRAFT_MODEL_KEY]
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("未检测到 GPU，请启用 GPU 运行时。")

random.seed(SPEC_SEED)
torch.manual_seed(SPEC_SEED)

if FREE_BASELINE_MODEL_BEFORE_SPEC:
    cleanup_names = [
        "hf_model", "autoreg_sampler", "naive_temp_output", "std_output",
        "mcmc_power_samp_output", "model_inputs", "input_ids", "naive_ids",
        "std_ids", "mcmc_generated_ids",
    ]
    for name in cleanup_names:
        if name in globals():
            del globals()[name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print_cuda_memory("after baseline cleanup")

json_path = "/content/MATH500.json"
if not os.path.isfile(json_path):
    raise FileNotFoundError("先运行下载 MATH500.json 的单元格")

with open(json_path, "r") as f:
    dataset = json.load(f)

start = 100 * SPEC_BATCH_IDX
end = 100 * (SPEC_BATCH_IDX + 1)
if SPEC_MAX_PROBLEMS is not None:
    end = min(start + int(SPEC_MAX_PROBLEMS), end)
print(f"spec shard [{start}, {end}) target={target_model_str} draft={draft_model_str} device={device}")

os.makedirs(SPEC_SAVE_DIR, exist_ok=True)
out_dir = os.path.join(SPEC_SAVE_DIR, "speculative_power")
os.makedirs(out_dir, exist_ok=True)

print("Loading shared processor/tokenizer...")
processor, tokenizer = load_text_processor(target_model_str)

print("Loading target model...")
target_model = load_generation_model(
    target_model_str,
    load_in_4bit=SPEC_TARGET_LOAD_IN_4BIT,
    load_in_8bit=SPEC_TARGET_LOAD_IN_8BIT,
)
print_cuda_memory("after target load")

print("Loading draft model...")
draft_model = load_generation_model(
    draft_model_str,
    load_in_4bit=SPEC_DRAFT_LOAD_IN_4BIT,
    load_in_8bit=SPEC_DRAFT_LOAD_IN_8BIT,
)
print_cuda_memory("after draft load")

device = infer_model_device(target_model)
target_vocab_size = model_vocab_size(target_model)
draft_vocab_size = model_vocab_size(draft_model)
tokenizer_size = len(tokenizer)
if target_vocab_size != draft_vocab_size:
    raise ValueError(
        "Target and draft vocab sizes differ; use models with the same tokenizer/vocabulary. "
        f"target={target_vocab_size}, draft={draft_vocab_size}, tokenizer={tokenizer_size}"
    )
if tokenizer_size > min(target_vocab_size, draft_vocab_size):
    raise ValueError(
        "Tokenizer can emit ids outside at least one model embedding matrix. "
        f"target={target_vocab_size}, draft={draft_vocab_size}, tokenizer={tokenizer_size}"
    )
print("Models loaded.")

def spec_sync_cuda():
    if str(device).startswith("cuda"):
        torch.cuda.synchronize()

spec_results = []
for data in tqdm(dataset[start:end], desc="MATH speculative power"):
    question = data["prompt"]
    answer = data["answer"]
    input_text = format_prompt(question, SPEC_TARGET_MODEL_KEY, tokenizer, SPEC_USE_COT)
    model_inputs = encode_text_prompt(processor, tokenizer, input_text, device)
    context_ids = model_inputs["input_ids"]

    spec_sync_cuda()
    t0 = time.perf_counter()
    generated_ids, spec_stats = run_speculative_power_sampling(
        target_model=target_model,
        draft_model=draft_model,
        tokenizer=tokenizer,
        context_ids=context_ids,
        alpha=SPEC_ALPHA,
        block_size=SPEC_BLOCK_SIZE,
        mcmc_steps_per_block=SPEC_MCMC_STEPS_PER_BLOCK,
        max_new_tokens=SPEC_MAX_NEW_TOKENS,
        draft_temperature=SPEC_DRAFT_TEMPERATURE,
        use_target_kv_cache=SPEC_USE_TARGET_KV_CACHE,
        validate_target_kv_cache_once=SPEC_VALIDATE_TARGET_KV_CACHE_ONCE,
    )
    spec_sync_cuda()
    seconds = time.perf_counter() - t0

    completion = tokenizer.decode(generated_ids, skip_special_tokens=True)
    predicted_answer = parse_answer(completion)
    spec_results.append({
        "question": question,
        "correct_answer": answer,
        "correct_answer_normalized": normalize_math_answer(answer),
        "spec_completion": completion,
        "spec_answer": predicted_answer,
        "spec_answer_normalized": normalize_math_answer(predicted_answer),
        "spec_correct": answers_match(predicted_answer, answer),
        "spec_tokens": len(generated_ids),
        "spec_seconds": seconds,
        "spec_tokens_per_second": len(generated_ids) / max(seconds, 1e-9),
        "spec_acceptance_ratio": spec_stats["acceptance_ratio"],
        "spec_attempts": spec_stats["attempts"],
        "spec_acceptances": spec_stats["acceptances"],
        "spec_used_target_kv_cache": spec_stats["used_target_kv_cache"],
        "spec_target_kv_cache_mode": spec_stats["target_kv_cache_mode"],
    })
    print(
        "spec_acceptance_ratio:", spec_stats["acceptance_ratio"],
        "correct:", answers_match(predicted_answer, answer),
        "tokens/sec:", len(generated_ids) / max(seconds, 1e-9),
    )

spec_csv_name = (
    f"spec_power_{SPEC_DRAFT_MODEL_KEY}_to_{SPEC_TARGET_MODEL_KEY}_"
    f"alpha{SPEC_ALPHA}_block{SPEC_BLOCK_SIZE}_steps{SPEC_MCMC_STEPS_PER_BLOCK}_"
    f"batch{SPEC_BATCH_IDX}_seed{SPEC_SEED}.csv"
)
spec_out_path = os.path.join(out_dir, spec_csv_name)
spec_df = pd.DataFrame(spec_results)
spec_df.to_csv(spec_out_path, index=False)
spec_metric_cols = ["spec_correct", "spec_acceptance_ratio", "spec_tokens_per_second", "spec_seconds", "spec_tokens"]
display(spec_df[spec_metric_cols].mean(numeric_only=True).to_frame("mean").T)
print("Saved:", spec_out_path)




In [ ]:
#@title 可选：比较 baseline 与 speculative 输出 CSV
import glob
import os
import pandas as pd

compare_root = "/content/rws_out"
baseline_files = sorted(glob.glob(os.path.join(compare_root, "qwen3_8b", "*.csv")))
spec_files = sorted(glob.glob(os.path.join(compare_root, "speculative_power", "*.csv")))
print("baseline files:", baseline_files[-3:])
print("speculative files:", spec_files[-3:])

frames = []
if baseline_files:
    base_df = pd.read_csv(baseline_files[-1])
    frames.append(pd.DataFrame([{
        "method": "baseline_mcmc_power",
        "correct": base_df["mcmc_correct"].mean() if "mcmc_correct" in base_df else None,
        "acceptance_ratio": base_df["acceptance_ratio"].mean() if "acceptance_ratio" in base_df else None,
        "tokens_per_second": base_df["mcmc_tokens_per_second"].mean() if "mcmc_tokens_per_second" in base_df else None,
        "seconds": base_df["mcmc_seconds"].mean() if "mcmc_seconds" in base_df else None,
        "tokens": base_df["mcmc_tokens"].mean() if "mcmc_tokens" in base_df else None,
    }]))
if spec_files:
    spec_df = pd.read_csv(spec_files[-1])
    frames.append(pd.DataFrame([{
        "method": "speculative_power",
        "correct": spec_df["spec_correct"].mean() if "spec_correct" in spec_df else None,
        "acceptance_ratio": spec_df["spec_acceptance_ratio"].mean() if "spec_acceptance_ratio" in spec_df else None,
        "tokens_per_second": spec_df["spec_tokens_per_second"].mean() if "spec_tokens_per_second" in spec_df else None,
        "seconds": spec_df["spec_seconds"].mean() if "spec_seconds" in spec_df else None,
        "tokens": spec_df["spec_tokens"].mean() if "spec_tokens" in spec_df else None,
    }]))

if frames:
    display(pd.concat(frames, ignore_index=True))
else:
    print("No CSVs found yet. Run the baseline and speculative cells first.")


## 可选：Google Drive

要保存到 Drive 时，先 `drive.mount('/content/drive')`，再把上面 `SAVE_DIR` 改为例如 `\/content/drive/MyDrive/rws_results`。
